# Binary Search

Objetivo: Encontrar um elemento em uma colecao **ordenada** eliminando metade dos candidatos a cada passo.

Complexidade:
- Tempo: O(log n) no pior caso, O(1) no melhor caso
- Espaco: O(1) iterativo | O(log n) recursivo (call stack)

Categoria: Busca | Tecnica: Divisao e Conquista

## Diagrama do fluxo

```mermaid
flowchart TD
    A[Inicio: low=0, high=n-1] --> B{low <= high ?}
    B -->|Nao| C[Retornar -1 nao encontrado]
    B -->|Sim| D[mid = low + high // 2]
    D --> E{arr mid == target ?}
    E -->|Sim| F[Retornar mid]
    E -->|Nao| G{arr mid < target ?}
    G -->|Sim| H[low = mid + 1  descartar metade esquerda]
    G -->|Nao| I[high = mid - 1  descartar metade direita]
    H --> B
    I --> B
```

## Fundamentos

Binary search explora a propriedade de dados ordenados para eliminar metade do espaco de busca a cada comparacao. Enquanto o linear search examina n elementos no pior caso, o binary search examina apenas log2(n) -- para 1 milhao de elementos, sao no maximo 20 comparacoes.

O pre-requisito e que a colecao esteja ordenada. Quando esse custo ja foi pago (por ordenacao previa, por uma estrutura que mantem a ordem, ou por dados naturalmente ordenados como timestamps), binary search e quase sempre a escolha correta.

Casos de uso reais incluem lookup em tabelas de geolocation por IP, busca em indices de banco de dados B-Tree, e localizacao de timestamps em series temporais de APIs.

In [ ]:
def binary_search_iterative(arr, target):
    """
    Searches for target in sorted array using iterative binary search.

    Args:
        arr: Sorted list of comparable elements.
        target: Value to search for.

    Returns:
        Index of target, or -1 if not found.

    Time:  O(log n)
    Space: O(1)  -- no call stack overhead
    """
    low, high = 0, len(arr) - 1

    while low <= high:
        # Compute midpoint safely (avoids integer overflow vs (low + high) // 2)
        mid = low + (high - low) // 2

        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            low = mid + 1   # target is in right half
        else:
            high = mid - 1  # target is in left half

    return -1


def binary_search_recursive(arr, target, low=0, high=None):
    """
    Searches for target in sorted array using recursive binary search.

    Args:
        arr: Sorted list of comparable elements.
        target: Value to search for.
        low: Lower bound index (inclusive).
        high: Upper bound index (inclusive).

    Returns:
        Index of target, or -1 if not found.

    Time:  O(log n)
    Space: O(log n)  -- recursive call stack
    """
    if high is None:
        high = len(arr) - 1

    if low > high:  # base case: search space is empty
        return -1

    mid = low + (high - low) // 2

    if arr[mid] == target:
        return mid
    elif arr[mid] < target:
        return binary_search_recursive(arr, target, mid + 1, high)
    else:
        return binary_search_recursive(arr, target, low, mid - 1)


# Validation
sorted_data = [2, 5, 8, 12, 16, 23, 38, 44, 56, 72, 91]
print(f"Array:                {sorted_data}")
print(f"Search 23 (iterative): index {binary_search_iterative(sorted_data, 23)}")   # 5
print(f"Search 23 (recursive): index {binary_search_recursive(sorted_data, 23)}")   # 5
print(f"Search 99 (iterative): index {binary_search_iterative(sorted_data, 99)}")   # -1
print(f"Search  2 (recursive): index {binary_search_recursive(sorted_data, 2)}")    # 0

## Variacoes: first occurrence, last occurrence e insertion point

O binary search padrao retorna **algum** indice quando o alvo aparece multiplas vezes. Nas variacoes abaixo implementamos:

- **find_first**: retorna o indice da primeira ocorrencia
- **find_last**: retorna o indice da ultima ocorrencia
- **find_insertion_point**: retorna a posicao onde o alvo deveria ser inserido para manter a ordenacao (equivalente a `bisect_left`)

In [ ]:
def find_first(arr, target):
    """
    Returns index of the FIRST occurrence of target in sorted arr.
    Returns -1 if not found.
    Time: O(log n)
    """
    low, high = 0, len(arr) - 1
    result = -1

    while low <= high:
        mid = low + (high - low) // 2
        if arr[mid] == target:
            result = mid       # record potential answer
            high = mid - 1    # keep looking LEFT for earlier occurrences
        elif arr[mid] < target:
            low = mid + 1
        else:
            high = mid - 1

    return result


def find_last(arr, target):
    """
    Returns index of the LAST occurrence of target in sorted arr.
    Returns -1 if not found.
    Time: O(log n)
    """
    low, high = 0, len(arr) - 1
    result = -1

    while low <= high:
        mid = low + (high - low) // 2
        if arr[mid] == target:
            result = mid       # record potential answer
            low = mid + 1     # keep looking RIGHT for later occurrences
        elif arr[mid] < target:
            low = mid + 1
        else:
            high = mid - 1

    return result


def find_insertion_point(arr, target):
    """
    Returns the leftmost index where target should be inserted
    to keep arr sorted (equivalent to bisect_left).
    Time: O(log n)
    """
    low, high = 0, len(arr)

    while low < high:
        mid = low + (high - low) // 2
        if arr[mid] < target:
            low = mid + 1
        else:
            high = mid

    return low


arr_with_dupes = [1, 3, 5, 5, 5, 7, 9, 11]
print(f"Array:               {arr_with_dupes}")
print(f"find_first(arr, 5):  {find_first(arr_with_dupes, 5)}")           # 2
print(f"find_last(arr, 5):   {find_last(arr_with_dupes, 5)}")            # 4
print(f"find_insertion_point(arr, 6): {find_insertion_point(arr_with_dupes, 6)}")  # 5
print(f"find_insertion_point(arr, 5): {find_insertion_point(arr_with_dupes, 5)}")  # 2

## Comparacao com o modulo bisect do Python

O modulo `bisect` da biblioteca padrao implementa binary search e insercao em listas ordenadas de forma eficiente em C. Compreender a implementacao manual facilita o uso correto do modulo.

In [ ]:
import bisect

arr = [10, 20, 30, 40, 50, 60, 70]

# bisect_left: insertion point on the left (same as find_insertion_point)
# bisect_right: insertion point on the right
# bisect.bisect_left(arr, x) == index of first element >= x
# bisect.bisect_right(arr, x) == index of first element > x

target = 40
idx_left  = bisect.bisect_left(arr, target)
idx_right = bisect.bisect_right(arr, target)
idx_manual = find_insertion_point(arr, target)

print(f"Array: {arr}")
print(f"Target: {target}")
print(f"bisect_left:         {idx_left}  -> arr[{idx_left}] = {arr[idx_left]}")
print(f"bisect_right:        {idx_right}  -> inserts AFTER existing {target}s")
print(f"find_insertion_point:{idx_manual}  (same as bisect_left)")

print()

# Practical: check membership using bisect
def bisect_contains(sorted_arr, target):
    """O(log n) membership test for sorted arrays using bisect."""
    idx = bisect.bisect_left(sorted_arr, target)
    return idx < len(sorted_arr) and sorted_arr[idx] == target

print(f"bisect_contains(arr, 40): {bisect_contains(arr, 40)}")  # True
print(f"bisect_contains(arr, 35): {bisect_contains(arr, 35)}")  # False

## Caso real: busca em faixas de IP para geolocalizacao

Bases de geolocalizacao de IP (como MaxMind GeoIP) armazenam faixas de enderecos IP em formato ordenado. Para descobrir o pais de um IP, e necessario encontrar a faixa que o contem -- um problema de busca em intervalos ordenados que o binary search resolve em O(log n).

Tambem demonstramos busca em timestamps de respostas de API para localizar um evento em uma serie temporal.

In [ ]:
from dataclasses import dataclass
import bisect
from datetime import datetime, timedelta


# --- IP Geolocation ---

@dataclass
class IpRange:
    start: int   # numeric representation of start IP
    end: int     # numeric representation of end IP
    country: str
    city: str


def ip_to_int(ip: str) -> int:
    """Converts dotted IPv4 to integer."""
    parts = ip.split(".")
    return (int(parts[0]) << 24) | (int(parts[1]) << 16) | (int(parts[2]) << 8) | int(parts[3])


# Simplified geolocation table (sorted by start IP)
GEO_RANGES = [
    IpRange(ip_to_int("1.0.0.0"),   ip_to_int("1.0.0.255"),   "AU", "Brisbane"),
    IpRange(ip_to_int("1.0.1.0"),   ip_to_int("1.0.3.255"),   "CN", "Fuzhou"),
    IpRange(ip_to_int("8.8.8.0"),   ip_to_int("8.8.8.255"),   "US", "Mountain View"),
    IpRange(ip_to_int("189.0.0.0"), ip_to_int("189.63.255.255"), "BR", "Sao Paulo"),
    IpRange(ip_to_int("200.0.0.0"), ip_to_int("200.255.255.255"), "BR", "Rio de Janeiro"),
]

START_IPS = [r.start for r in GEO_RANGES]  # sorted list of start IPs


def geolocate(ip: str) -> str | None:
    """
    Returns country/city for an IP address using binary search on sorted IP ranges.
    Time: O(log n) where n is number of IP ranges.
    """
    ip_int = ip_to_int(ip)
    # bisect_right gives first range whose start > ip_int
    # so the candidate range is one position to the left
    pos = bisect.bisect_right(START_IPS, ip_int) - 1

    if pos < 0:
        return None

    candidate = GEO_RANGES[pos]
    if candidate.start <= ip_int <= candidate.end:
        return f"{candidate.city}, {candidate.country}"
    return None


test_ips = ["8.8.8.1", "189.50.10.5", "1.0.2.100", "192.168.1.1"]
print("=== IP Geolocation ===")
for ip in test_ips:
    location = geolocate(ip)
    print(f"  {ip:<18} -> {location or 'not found'}")


# --- Binary search on sorted API timestamps ---

print("\n=== Timestamp search in API response series ===")

base_ts = datetime(2024, 3, 1, 10, 0, 0)
# Simulate 500 API response timestamps at 5-second intervals
timestamps = [base_ts + timedelta(seconds=5 * i) for i in range(500)]
ts_numeric = [ts.timestamp() for ts in timestamps]  # sorted floats

target_ts = base_ts + timedelta(minutes=20, seconds=35)  # exactly at index 247
idx = binary_search_iterative(ts_numeric, target_ts.timestamp())
print(f"Target timestamp: {target_ts}")
print(f"Found at index:   {idx}")
print(f"Confirmed entry:  {timestamps[idx]}")

## Analise de Complexidade

### Tempo

**Melhor caso: O(1)** - O elemento do meio e o alvo na primeira iteracao.

**Caso medio e pior caso: O(log n)** - A cada iteracao o espaco de busca e dividido ao meio. Para n=1.000.000 sao no maximo log2(1.000.000) ≈ 20 iteracoes.

### Espaco

**O(1)** na versao iterativa -- apenas variaveis `low`, `high`, `mid`.

**O(log n)** na versao recursiva -- a call stack ocupa espaco proporcional a profundidade da recursao.

### Pre-requisito: dados ordenados

O custo de ordenar uma colecao e O(n log n). Se uma busca unica for realizada, linear search O(n) pode ser mais rapido no total. Binary search compensa quando:
- Os dados ja estao ordenados (timestamps, IDs sequenciais)
- Multiplas buscas serao realizadas na mesma colecao
- A colecao e mantida ordenada por uma estrutura de dados (B-tree, sorted set)

In [ ]:
import timeit
import random

SIZES = [10_000, 100_000, 1_000_000]
RUNS = 50

print(f"{'n':>12} | {'linear (ms)':>13} | {'binary (ms)':>13} | {'speedup':>10}")
print("-" * 57)

for n in SIZES:
    data = sorted(random.sample(range(n * 10), n))
    target = random.choice(data)  # guaranteed to exist

    # Worst-case linear: target at end -- use a missing value that forces full scan
    missing = max(data) + 1

    t_linear = timeit.timeit(
        lambda d=data, t=missing: next((i for i, v in enumerate(d) if v == t), -1),
        number=RUNS
    ) / RUNS * 1000

    t_binary = timeit.timeit(
        lambda d=data, t=target: binary_search_iterative(d, t),
        number=RUNS
    ) / RUNS * 1000

    speedup = t_linear / t_binary if t_binary > 0 else float("inf")
    print(f"{n:>12,} | {t_linear:>13.4f} | {t_binary:>13.4f} | {speedup:>9.1f}x")

## Comparacao com alternativas

| Algoritmo | Complexidade | Pre-requisito | Caso de uso |
|---|---|---|---|
| Linear Search | O(n) | Nenhum | Dados nao ordenados, colecoes pequenas |
| Binary Search | O(log n) | Dados ordenados | Grandes colecoes ordenadas |
| Hash Table (dict) | O(1) medio | Dados hashaveis | Buscas frequentes por chave exata |
| B-Tree index | O(log n) | Construcao previa | Bancos de dados, range queries |
| Trie | O(k) onde k = len(key) | Strings | Busca por prefixo |

Para **busca por chave exata** com muitas consultas, `dict` (hash table) supera binary search. Binary search brilha em **range queries** e quando os dados ja estao ordenados sem possibilidade de hash.

## Exercicios

**Desafio 1:** Implemente `lower_bound(arr, target)` e `upper_bound(arr, target)` equivalentes ao `bisect_left` e `bisect_right` respectivamente, sem usar o modulo `bisect`. Valide contra as funcoes do modulo.

**Desafio 2:** Implemente `search_rotated_sorted(arr, target)` que busca um elemento em um array **ordenado e rotacionado** (ex: `[15, 18, 2, 3, 6, 12]`). Mantenha complexidade O(log n).

In [ ]:
import bisect

# Desafio 1: lower_bound e upper_bound
def lower_bound(arr, target):
    """
    Returns the leftmost index where target can be inserted to keep arr sorted.
    Equivalent to bisect.bisect_left.
    """
    low, high = 0, len(arr)
    while low < high:
        mid = low + (high - low) // 2
        if arr[mid] < target:
            low = mid + 1
        else:
            high = mid
    return low


def upper_bound(arr, target):
    """
    Returns the rightmost index where target can be inserted to keep arr sorted.
    Equivalent to bisect.bisect_right.
    """
    low, high = 0, len(arr)
    while low < high:
        mid = low + (high - low) // 2
        if arr[mid] <= target:
            low = mid + 1
        else:
            high = mid
    return low


arr = [1, 3, 5, 5, 5, 7, 9]
for t in [5, 4, 9, 0]:
    lb = lower_bound(arr, t)
    ub = upper_bound(arr, t)
    assert lb == bisect.bisect_left(arr, t), f"lower_bound mismatch for {t}"
    assert ub == bisect.bisect_right(arr, t), f"upper_bound mismatch for {t}"
    print(f"target={t}: lower_bound={lb}, upper_bound={ub}  (count={ub-lb})")

print()

# Desafio 2: busca em array rotacionado
def search_rotated_sorted(arr, target):
    """
    Binary search on a rotated sorted array.
    One half is always sorted; determine which and discard accordingly.
    Time: O(log n)
    """
    low, high = 0, len(arr) - 1

    while low <= high:
        mid = low + (high - low) // 2

        if arr[mid] == target:
            return mid

        # Left half is sorted
        if arr[low] <= arr[mid]:
            if arr[low] <= target < arr[mid]:
                high = mid - 1
            else:
                low = mid + 1
        # Right half is sorted
        else:
            if arr[mid] < target <= arr[high]:
                low = mid + 1
            else:
                high = mid - 1

    return -1


rotated = [15, 18, 2, 3, 6, 12]
print(f"Rotated array: {rotated}")
for t in [6, 15, 18, 1]:
    print(f"  search_rotated_sorted(arr, {t}) = {search_rotated_sorted(rotated, t)}")

## Referencias

1. Knuth, D. E. "The Art of Computer Programming, Vol. 3: Sorting and Searching" (2nd ed.). Addison-Wesley, 1998, pp. 406-423.
2. Cormen, T. H., et al. "Introduction to Algorithms" (3rd ed.). MIT Press, 2009, pp. 798-799.
3. Python Software Foundation. "bisect - Array bisection algorithm". Python 3 docs. https://docs.python.org/3/library/bisect.html